# Harmony Project 3 — LoRA Adapter Evaluation

In [ ]:
!pip install -q "transformers==4.46.3" "trl==0.11.4" peft bitsandbytes accelerate datasets json-repair

In [ ]:
import os, sys, json, time, torch
from pathlib import Path

BASE_MODEL          = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
ADAPTER_PATH        = "/kaggle/input/harmony-lora-adapter/adapter"
TEST_FILE           = "/kaggle/input/datasets/keertanaks2004/harmony-ade-data/test.jsonl"
OOD_FILE            = None   # set to path if OOD jsonl uploaded
OUTPUT_DIR          = "/kaggle/working/eval_reports"
MAX_SAMPLES         = None   # set to 50 for quick test, None for full eval
SANITY_N            = 10     # examples for sanity check

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify files
assert Path(ADAPTER_PATH).exists(), f"Adapter not found: {ADAPTER_PATH}"
assert Path(TEST_FILE).exists(), f"Test file not found: {TEST_FILE}"
assert torch.cuda.is_available(), "No GPU — this notebook requires T4"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Test file found. Config OK.")

In [ ]:
# ============================================================
# app/schemas/extraction.py — inlined
# ============================================================
# app/schemas/extraction.py
#
# Project 3 — Phase 2: Pydantic v2 schema for clinical structured extraction.
#
# IMPORTANT — two validators with similar names:
#   app/ingestion/extraction_validator.py  ->  Project 1, OCR quality scorer. DO NOT TOUCH.
#   app/ingestion/validator.py             ->  Project 3, JSON repair + Pydantic wrapper (this phase).
#
# Design decisions:
#   D-06: Hybrid schema — strict enums for entity_type/relation_status, free text for mention/evidence.
#   D-07: Schema scope is drug + ADE + dosage + relation + evidence span. No assertion_status etc.
#   D-35: record_id and validation are NOT model outputs. They are system-injected by
#         app/ingestion/validator.py after generation. Never include them in training targets.
#
# Schema version history:
#   v1 (2026): Initial. entity_type in {medication, adverse_event}. relation_status in {related,
#              not_related, none}. No assertion_status, certainty, temporal_status (not labeled
#              in ade_corpus_v2 — lead-approved scope reduction).

from typing import Literal, Optional
from pydantic import BaseModel, Field, model_validator


class SourceSpan(BaseModel):
    """Character-level evidence span in the input text.

    start_char and end_char are offsets into the ORIGINAL unmasked text
    (extraction runs before PHI masking per D-35).
    """

    start_char: int = Field(ge=0, description="Start character offset (inclusive).")
    end_char: int = Field(ge=0, description="End character offset (exclusive).")


class Entity(BaseModel):
    """A single extracted clinical entity — medication or adverse event.

    Fields:
        entity_type       : Strict enum. medication | adverse_event.
        mention           : The exact surface form found in the text. Free text.
        dosage            : Dosage string (e.g. "500 mg BID"). Null if not present or not a medication.
        linked_medication : For adverse_event entities — the drug this ADE is linked to.
                            Null for medication entities. Free text (drug mention string).
        evidence          : Supporting sentence text. Should be a substring of the input chunk.
        source_span       : Character offsets of `mention` in the input text.
    """

    entity_type: Literal["medication", "adverse_event"]
    mention: str = Field(min_length=1, description="Surface form of the entity in the source text.")
    dosage: Optional[str] = None
    linked_medication: Optional[str] = None
    evidence: str = Field(min_length=1, description="Sentence-level supporting evidence text.")
    source_span: SourceSpan


class ValidationFlags(BaseModel):
    """Post-generation validation state — system-injected, never model-generated.

    All flags start True and are flipped to False if the corresponding check fails.
    json_valid is False only when JSON repair also fails (raw output is unparseable).
    schema_valid is False when Pydantic model_validate() raises ValidationError.
    enum_valid is False when entity_type or relation_status contain invalid values.
        (In practice Pydantic v2 Literal already enforces this, but the flag is explicit
        so downstream consumers don't need to inspect the Pydantic error.)
    evidence_present is False when any entity's evidence field is not a substring of
        the input chunk text. The extraction is still returned — not rejected — but flagged.
    """

    json_valid: bool
    schema_valid: bool
    enum_valid: bool
    evidence_present: bool


class ExtractionResult(BaseModel):
    """Full system-level extraction result, after generation + validation.

    Split into two parts (see Design Doc §10.1):

    MODEL-GENERATED (model is trained to produce these):
        schema_version, entities[], relation_status

    SYSTEM-INJECTED (wrapper adds these BEFORE model_validate()):
        record_id, validation, error_reason

    Never put record_id or validation in training targets — doing so would teach the model
    to invent record IDs and claim its own output is always valid.
    """

    record_id: str = Field(description="Chunk ID — injected by validator.py, not by the model.")
    schema_version: Literal["v1"]
    entities: list[Entity]
    relation_status: Literal["related", "not_related", "none"]
    validation: ValidationFlags
    error_reason: Optional[str] = None  # Populated by build_empty_result() on failure; None on success.

    @model_validator(mode="after")
    def check_spans_make_sense(self) -> "ExtractionResult":
        """Reject any entity whose source_span has end_char <= start_char.

        A zero-length or inverted span is always a model error (the model copied wrong offsets).
        We catch this here rather than silently storing broken span data in OpenSearch.
        """
        for entity in self.entities:
            span = entity.source_span
            if span.end_char <= span.start_char:
                raise ValueError(
                    f"Invalid source_span for entity '{entity.mention}': "
                    f"end_char ({span.end_char}) must be > start_char ({span.start_char})."
                )
        return self


# ============================================================
# app/ingestion/validator.py — inlined
# ============================================================
# app/ingestion/validator.py
#
# Project 3 — Phase 2: JSON repair + Pydantic validation wrapper.
#
# DO NOT CONFUSE with app/ingestion/extraction_validator.py (Project 1 OCR quality scorer).
# These are completely different files with different purposes.
#
# This module is the validation engine described in Design Doc §15.
# It is called by app/ingestion/extractor.py (Phase 6) after the fine-tuned model generates
# a raw string, and also used directly by the evaluation harness (Phase 5).
#
# Validation pipeline (Design Doc §15.1, layers 2–4):
#   Layer 2: json.loads() -> json_repair.loads() fallback
#   Layer 3: ExtractionResult.model_validate() (Pydantic v2)
#   Layer 4: Evidence substring check + hallucination warning
#
# Requires: pip install json-repair

import json
import logging
from typing import Optional

from pydantic import ValidationError

logger = logging.getLogger(__name__)


def build_empty_result(record_id: str, reason: str) -> ExtractionResult:
    """Return a fully-structured but empty ExtractionResult indicating a failure.

    Used when JSON parsing fails, schema validation fails, or extraction is disabled.
    All validation flags are set to False. error_reason explains the failure.

    Args:
        record_id : The chunk ID this extraction was attempted for.
        reason    : Short machine-readable failure code. Common values:
                    "json_parse_failed"     -- json.loads + json_repair both failed
                    "schema_invalid"        -- Pydantic model_validate() raised ValidationError
                    "extraction_disabled"   -- EXTRACTION_ENABLED env var is "false"
                    "extraction_error"      -- Unexpected exception (OOM, CUDA error, etc.)

    Returns:
        ExtractionResult with empty entities, relation_status="none", all flags False.
    """
    return ExtractionResult(
        record_id=record_id,
        schema_version="v1",
        entities=[],
        relation_status="none",
        validation=ValidationFlags(
            json_valid=False,
            schema_valid=False,
            enum_valid=False,
            evidence_present=False,
        ),
        error_reason=reason,
    )


def validate_extraction(
    raw_text: str,
    record_id: str,
    input_text: str,
) -> ExtractionResult:
    """Validate and repair the raw string output from the fine-tuned model.

    Steps (Design Doc §15.1, Layers 2–4):
      1. Try json.loads(raw_text).
      2. On JSONDecodeError: try json_repair.loads(raw_text).
         If still fails -> return build_empty_result(reason="json_parse_failed").
      3. Inject system fields: record_id + validation block.
         Both MUST be injected before model_validate() — ExtractionResult requires them.
      4. ExtractionResult.model_validate(parsed).
         On ValidationError -> return build_empty_result(reason="schema_invalid").
      5. Evidence substring check: for each entity, verify evidence is a substring of input_text.
         If not, flip validation.evidence_present = False (flagged, not rejected).
      6. Hallucination check: for each entity, verify mention is a substring of input_text.
         If not, log a warning (hallucination_warnings list for offline analysis).
      7. Return the validated ExtractionResult.

    Args:
        raw_text   : Raw string from the model (may be broken JSON).
        record_id  : Chunk ID to inject as record_id.
        input_text : Original (unmasked) chunk text, used for evidence and mention checks.

    Returns:
        ExtractionResult. Always returns a valid object — never raises.
    """
    # -- Layer 2: JSON parse -----------------------------------------------
    parsed: Optional[dict] = None
    json_valid = True

    try:
        parsed = json.loads(raw_text)
    except (json.JSONDecodeError, ValueError):
        json_valid = False
        try:
            import json_repair  # noqa: PLC0415  (late import — optional dependency)
            parsed = json_repair.loads(raw_text)
            if not isinstance(parsed, dict):
                logger.warning(
                    "record_id=%s json_repair returned non-dict type %s — treating as failure",
                    record_id,
                    type(parsed).__name__,
                )
                return build_empty_result(record_id, reason="json_parse_failed")
            logger.info("record_id=%s JSON repaired successfully", record_id)
        except Exception as repair_exc:  # noqa: BLE001
            logger.error(
                "record_id=%s JSON repair failed: %s", record_id, repair_exc
            )
            return build_empty_result(record_id, reason="json_parse_failed")

    if not isinstance(parsed, dict):
        logger.warning(
            "record_id=%s json.loads returned non-dict type %s", record_id, type(parsed).__name__
        )
        return build_empty_result(record_id, reason="json_parse_failed")

    # -- Layer 3: Inject system fields + Pydantic validation ----------------
    # BOTH record_id AND validation must be injected here.
    # The model never outputs these — they are system-injected per Design Doc D-35.
    parsed["record_id"] = record_id
    parsed["validation"] = {
        "json_valid": json_valid,
        "schema_valid": True,   # will stay True if model_validate succeeds
        "enum_valid": True,     # Literal enforcement by Pydantic catches bad enum values
        "evidence_present": True,  # updated in Layer 4 below
    }

    try:
        result: ExtractionResult = ExtractionResult.model_validate(parsed)
    except ValidationError as ve:
        logger.warning(
            "record_id=%s Pydantic validation failed: %s", record_id, ve
        )
        return build_empty_result(record_id, reason="schema_invalid")

    # -- Layer 4: Evidence substring check + hallucination detection --------
    hallucination_warnings: list[str] = []

    for entity in result.entities:
        # Evidence check: the supporting sentence should be a substring of the input chunk.
        if entity.evidence and entity.evidence not in input_text:
            result.validation.evidence_present = False
            logger.debug(
                "record_id=%s entity '%s' evidence not substring of input",
                record_id,
                entity.mention,
            )

        # Hallucination check: the entity mention should appear in the input text.
        if entity.mention and entity.mention not in input_text:
            hallucination_warnings.append(entity.mention)
            logger.warning(
                "record_id=%s HALLUCINATION WARNING: mention '%s' not found in input text",
                record_id,
                entity.mention,
            )

    if hallucination_warnings:
        logger.warning(
            "record_id=%s %d hallucinated mention(s): %s",
            record_id,
            len(hallucination_warnings),
            hallucination_warnings,
        )

    return result


print("Schema and validator inlined OK.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model (FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16, device_map="auto",
)
model.config.use_cache = False

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

print(f"Model ready. GPU 0: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB free")

In [ ]:
def run_inference(text: str) -> str:
    """Run model on a clinical text, return raw string output."""
    messages = [{"role": "user", "content": (
        "You are a clinical information extractor. Given a clinical text, extract all\n"
        "medications and adverse events as a JSON object that follows the schema below.\n"
        "Return ONLY valid JSON. If no entity is present, return entities=[] and\n"
        "relation_status=\"none\".\n\n"
        "Return ONLY this JSON structure (no record_id, no validation block — those are added by the system):\n"
        "{\n  \"schema_version\": \"v1\",\n  \"entities\": [\n    {\n"
        "      \"entity_type\": \"medication\" | \"adverse_event\",\n"
        "      \"mention\": \"<string>\",\n      \"dosage\": \"<string>\" | null,\n"
        "      \"linked_medication\": \"<string>\" | null,\n      \"evidence\": \"<string>\",\n"
        "      \"source_span\": {\"start_char\": <int>, \"end_char\": <int>}\n    }\n  ],\n"
        "  \"relation_status\": \"related\" | \"not_related\" | \"none\"\n}\n\n"
        f"Clinical text:\n{text}"
    )}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, do_sample=False, max_new_tokens=512,
            repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return raw.strip()

# Load test data
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

test_data = load_jsonl(TEST_FILE)
print(f"Test set: {len(test_data)} examples")

# Sanity check on SANITY_N examples
print(f"\n{'='*60}")
print(f"SANITY CHECK — {SANITY_N} examples")
print(f"{'='*60}")

json_ok = 0
schema_ok = 0
for i, row in enumerate(test_data[:SANITY_N]):
    user_content = row["messages"][0]["content"]
    input_text = user_content.split("Clinical text:\n", 1)[1].strip()
    gold_raw = row["messages"][1]["content"]
    
    t0 = time.time()
    raw_output = run_inference(input_text)
    latency = time.time() - t0
    
    result = validate_extraction(raw_output, record_id=f"test_{i}", input_text=input_text)
    
    if result.validation.json_valid: json_ok += 1
    if result.validation.schema_valid: schema_ok += 1
    
    print(f"\n--- Example {i+1} ---")
    print(f"Input: {input_text[:80]}...")
    print(f"Raw output: {raw_output[:120]}...")
    print(f"JSON valid: {result.validation.json_valid} | Schema valid: {result.validation.schema_valid}")
    print(f"Entities extracted: {len(result.entities)} | Latency: {latency:.1f}s")
    if result.entities:
        for e in result.entities[:2]:
            print(f"  [{e.entity_type}] {e.mention!r}  dosage={e.dosage}")

print(f"\nSanity check: {json_ok}/{SANITY_N} JSON valid, {schema_ok}/{SANITY_N} schema valid")

In [ ]:
import numpy as np
from collections import defaultdict

samples = test_data if MAX_SAMPLES is None else test_data[:MAX_SAMPLES]
print(f"Running full eval on {len(samples)} examples...")

# Accumulators
results_list = []
gold_entities_list = []
pred_relation_statuses = []
gold_relation_statuses = []
latencies = []
json_valid_pre = 0   # valid with json.loads directly (before repair)
errors = []          # for error_analysis

for i, row in enumerate(samples):
    if i % 100 == 0:
        print(f"  {i}/{len(samples)}...")
    
    user_content = row["messages"][0]["content"]
    input_text = user_content.split("Clinical text:\n", 1)[1].strip()
    gold_raw = row["messages"][1]["content"]
    gold_dict = json.loads(gold_raw)
    
    t0 = time.time()
    raw_output = run_inference(input_text)
    latency = time.time() - t0
    latencies.append(latency)
    
    # Track pre-repair JSON validity
    try:
        json.loads(raw_output)
        json_valid_pre += 1
    except:
        pass
    
    result = validate_extraction(raw_output, record_id=f"test_{i}", input_text=input_text)
    
    # Build gold entities
    gold_entities = []
    for e in gold_dict.get("entities", []):
        try:
            gold_entities.append(Entity(
                entity_type=e["entity_type"], mention=e["mention"],
                dosage=e.get("dosage"), linked_medication=e.get("linked_medication"),
                evidence=e["evidence"],
                source_span=SourceSpan(start_char=e["source_span"]["start_char"], end_char=e["source_span"]["end_char"])
            ))
        except:
            pass
    
    results_list.append(result)
    gold_entities_list.append(gold_entities)
    pred_relation_statuses.append(result.relation_status)
    gold_relation_statuses.append(gold_dict.get("relation_status", "none"))
    
    # Collect errors
    if len(errors) < 20:
        error_type = None
        if not result.validation.json_valid: error_type = "json_invalid"
        elif not result.validation.schema_valid: error_type = "schema_invalid"
        elif not result.validation.evidence_present: error_type = "span_error"
        elif any(e.mention not in input_text for e in result.entities): error_type = "hallucination"
        if error_type:
            errors.append({"id": f"test_{i}", "input": input_text[:200], "predicted": raw_output[:300], "gold": gold_raw[:300], "error_type": error_type})

print("Inference complete. Computing metrics...")

# ── Inline metric helpers (self-contained, no import from evaluation/) ──────────

def compute_json_validity_rate(results):
    """Fraction of results where json_valid=True (post-repair)."""
    return sum(1 for r in results if r.validation.json_valid) / len(results) if results else 0.0

def compute_schema_validity_rate(results):
    """Fraction of results where schema_valid=True."""
    return sum(1 for r in results if r.validation.schema_valid) / len(results) if results else 0.0

def compute_enum_accuracy(results):
    """Fraction of results where enum_valid=True."""
    return sum(1 for r in results if r.validation.enum_valid) / len(results) if results else 0.0

def compute_entity_f1(pred_entities, gold_entities, entity_type: str) -> dict:
    """Compute TP/FP/FN for mention-level exact match for a given entity_type."""
    pred_mentions = {e.mention.lower().strip() for e in pred_entities if e.entity_type == entity_type}
    gold_mentions = {e.mention.lower().strip() for e in gold_entities if e.entity_type == entity_type}
    tp = len(pred_mentions & gold_mentions)
    fp = len(pred_mentions - gold_mentions)
    fn = len(gold_mentions - pred_mentions)
    return {"tp": tp, "fp": fp, "fn": fn}

def compute_hallucination_rate(pred_entities, input_text: str) -> float:
    """Fraction of predicted entities whose mention is NOT in the input text."""
    if not pred_entities:
        return 0.0
    hallucinated = sum(1 for e in pred_entities if e.mention not in input_text)
    return hallucinated / len(pred_entities)

def compute_evidence_accuracy(pred_entities, input_text: str) -> float:
    """Fraction of predicted entities whose evidence IS a substring of input_text."""
    if not pred_entities:
        return 1.0  # no entities → no evidence errors
    correct = sum(1 for e in pred_entities if e.evidence and e.evidence in input_text)
    return correct / len(pred_entities)

def compute_relation_f1(pred_statuses, gold_statuses) -> dict:
    """Macro-averaged F1 across the three relation_status classes."""
    classes = ["related", "not_related", "none"]
    f1s = []
    for cls in classes:
        tp = sum(1 for p, g in zip(pred_statuses, gold_statuses) if p == cls and g == cls)
        fp = sum(1 for p, g in zip(pred_statuses, gold_statuses) if p == cls and g != cls)
        fn = sum(1 for p, g in zip(pred_statuses, gold_statuses) if p != cls and g == cls)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return {"macro_f1": float(np.mean(f1s)), "per_class_f1": dict(zip(classes, f1s))}

def compute_aggregate_span_metrics(result, gold_entities, input_text: str) -> dict:
    """Per-example strict and lenient (IoU >= 0.5) span match counts."""
    strict_correct = 0
    lenient_correct = 0
    total = 0
    for pred_e in result.entities:
        # Find best matching gold entity by mention (case-insensitive)
        matches = [g for g in gold_entities
                   if g.entity_type == pred_e.entity_type
                   and g.mention.lower().strip() == pred_e.mention.lower().strip()]
        if not matches:
            total += 1
            continue
        gold_e = matches[0]
        ps, pe = pred_e.source_span.start_char, pred_e.source_span.end_char
        gs, ge = gold_e.source_span.start_char, gold_e.source_span.end_char
        total += 1
        if ps == gs and pe == ge:
            strict_correct += 1
        # IoU
        inter = max(0, min(pe, ge) - max(ps, gs))
        union = max(pe, ge) - min(ps, gs)
        iou = inter / union if union > 0 else 0.0
        if iou >= 0.5:
            lenient_correct += 1
    return {"strict_span_correct": strict_correct, "lenient_span_correct": lenient_correct, "span_total": total}


# Aggregate metrics
all_pred_drug_tp = all_pred_drug_fp = all_pred_drug_fn = 0
all_pred_ade_tp = all_pred_ade_fp = all_pred_ade_fn = 0
span_strict_correct = span_lenient_correct = span_total = 0
hall_rates = []
evid_rates = []

for result, gold_entities in zip(results_list, gold_entities_list):
    d = compute_entity_f1(result.entities, gold_entities, "medication")
    all_pred_drug_tp += d["tp"]; all_pred_drug_fp += d["fp"]; all_pred_drug_fn += d["fn"]
    
    a = compute_entity_f1(result.entities, gold_entities, "adverse_event")
    all_pred_ade_tp += a["tp"]; all_pred_ade_fp += a["fp"]; all_pred_ade_fn += a["fn"]
    
    # use empty string for input_text in span metrics (spans are index-based, not substring)
    span_m = compute_aggregate_span_metrics(result, gold_entities, "")
    span_strict_correct += span_m["strict_span_correct"]
    span_lenient_correct += span_m["lenient_span_correct"]
    span_total += span_m["span_total"]
    
    # retrieve input_text for this example
    _idx = results_list.index(result)
    _user_content = samples[_idx]["messages"][0]["content"]
    _input_text = _user_content.split("Clinical text:\n", 1)[1].strip()
    hall_rates.append(compute_hallucination_rate(result.entities, _input_text))
    evid_rates.append(compute_evidence_accuracy(result.entities, _input_text))

def safe_f1(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp>0 else 0.0
    r = tp/(tp+fn) if tp+fn>0 else 0.0
    return 2*p*r/(p+r) if p+r>0 else 0.0, p, r

drug_f1, drug_p, drug_r = safe_f1(all_pred_drug_tp, all_pred_drug_fp, all_pred_drug_fn)
ade_f1, ade_p, ade_r = safe_f1(all_pred_ade_tp, all_pred_ade_fp, all_pred_ade_fn)
rel_metrics = compute_relation_f1(pred_relation_statuses, gold_relation_statuses)
lat = np.array(latencies)

metrics = {
    "json_valid_pre_repair": round(json_valid_pre/len(samples), 4),
    "json_valid_post_repair": round(compute_json_validity_rate(results_list), 4),
    "schema_valid": round(compute_schema_validity_rate(results_list), 4),
    "drug_f1": round(drug_f1, 4), "drug_precision": round(drug_p, 4), "drug_recall": round(drug_r, 4),
    "ade_f1": round(ade_f1, 4), "ade_precision": round(ade_p, 4), "ade_recall": round(ade_r, 4),
    "relation_f1": round(rel_metrics["macro_f1"], 4),
    "hallucination_rate": round(float(np.mean(hall_rates)), 4),
    "evidence_accuracy": round(float(np.mean(evid_rates)), 4),
    "enum_accuracy": round(compute_enum_accuracy(results_list), 4),
    "span_f1_strict": round(span_strict_correct/span_total, 4) if span_total>0 else 0.0,
    "span_f1_lenient": round(span_lenient_correct/span_total, 4) if span_total>0 else 0.0,
    "latency_p50_s": round(float(np.percentile(lat, 50)), 3),
    "latency_p95_s": round(float(np.percentile(lat, 95)), 3),
}

print("\n" + "="*60)
print("EVALUATION RESULTS — lora_v1")
print("="*60)
targets = {
    "json_valid_pre_repair": (">=95%", 0.95), "json_valid_post_repair": (">=99.5%", 0.995),
    "schema_valid": (">=90%", 0.90), "drug_f1": (">=75%", 0.75), "ade_f1": (">=65%", 0.65),
    "relation_f1": (">=70%", 0.70), "hallucination_rate": ("<=5%", None),
    "evidence_accuracy": (">=90%", 0.90), "span_f1_strict": (">=65%", 0.65),
    "span_f1_lenient": (">=75%", 0.75),
}
for k, v in metrics.items():
    target_str, target_val = targets.get(k, ("", None))
    if target_val is not None:
        if k == "hallucination_rate":
            ok = "PASS" if v <= 0.05 else "FAIL"
        else:
            ok = "PASS" if v >= target_val else "FAIL"
        print(f"  [{ok}] {k:<35} {v:.4f}  (target {target_str})")
    else:
        print(f"        {k:<35} {v}")

In [ ]:
import datetime

report = {
    "model": "lora",
    "adapter_path": ADAPTER_PATH,
    "test_file": TEST_FILE,
    "n_examples": len(samples),
    "metrics": metrics,
    "error_analysis": errors,
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
}

out_path = f"{OUTPUT_DIR}/lora_v1.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {out_path}")
print(f"\nDownload {OUTPUT_DIR}/ before session ends.")

## Done

Download `/kaggle/working/eval_reports/lora_v1.json` when eval finishes.

Move it to `evaluation/reports/lora_v1.json` in the project.